# Grafos, centralidades e comunidades

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_06/04_grafos_centralidade_e_comunidades.ipynb)

Um grafo reúne nós e arestas; ambos podem ter atributos, direção e peso. Grau
conta vizinhos, força soma pesos e intermediação resume participação em caminhos
mínimos. Cada medida responde a uma pergunta estrutural distinta.

![Grau, força e intermediação destacam propriedades diferentes do mesmo grafo.](imagens/04_centralidades.svg)

## 1. Da tabela de arestas ao grafo

A célula seguinte lê `arestas_coocorrencia.csv`. Cada linha é um par de entidades
e `peso` é o número de documentos compartilhados. O NetworkX transforma essa
tabela num grafo não dirigido; ele não acrescenta significado histórico à aresta.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_06'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('networkx', 'networkx>=3,<4')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

arestas = pd.read_csv("dados/arestas_coocorrencia.csv")
grafo = nx.from_pandas_edgelist(
    arestas, source="origem", target="destino", edge_attr="peso"
)
print("Nós:", grafo.number_of_nodes(), "Arestas:", grafo.number_of_edges())

## 2. Medidas como perguntas

- **grau:** com quantos nós diferentes esta entidade coocorre?
- **força:** quantas coocorrências ponderadas incidem sobre ela?
- **intermediação:** em quantos caminhos mínimos ela ocupa posição intermediária?

Para um grafo não dirigido com $n$ nós, o grau normalizado é
$C_D(v)=deg(v)/(n-1)$; a força é $s(v)=\sum_u w_{vu}$. A intermediação é:

$$C_B(v)=\sum_{s\ne v\ne t}\frac{\sigma_{st}(v)}{\sigma_{st}}$$

$s$ e $t$ são os extremos; $\sigma_{st}$ conta caminhos mínimos entre eles e
$\sigma_{st}(v)$ conta os que passam por $v$. As fórmulas deixam claro que
vizinhos, repetição de vínculos e posição de ponte são propriedades diferentes.

Peso de coocorrência representa repetição documental, não confiança, influência
ou intensidade social. Como o peso aqui representa **força**, não distância, a
intermediação abaixo usa `weight=None`. Usar o peso diretamente faria uma ligação
mais frequente parecer um caminho mais longo; uma análise ponderada exigiria
transformar força em distância e justificar essa operação.

In [ ]:
grau = nx.degree_centrality(grafo)
forca = dict(grafo.degree(weight="peso"))
intermediacao = nx.betweenness_centrality(grafo, weight=None)
metricas = pd.DataFrame({
    "grau_normalizado": pd.Series(grau),
    "forca": pd.Series(forca),
    "intermediacao": pd.Series(intermediacao),
}).sort_values("forca", ascending=False)
display(metricas.round(3))

comunidades = list(nx.community.greedy_modularity_communities(grafo, weight="peso"))
comunidade_de = {no: i + 1 for i, grupo in enumerate(comunidades) for no in grupo}
metricas["comunidade_algoritmica"] = pd.Series(comunidade_de)
display(metricas.sort_values(["comunidade_algoritmica", "forca"]))

Comunidades maximizam uma função estrutural sob escolhas algorítmicas; não são
automaticamente grupos sociais reconhecidos. Estabilidade a recortes e retorno
aos documentos são necessários antes de nomeá-las.

In [ ]:
posicao = nx.spring_layout(grafo, seed=2026, weight="peso")
tamanhos = [250 + 55 * forca[no] for no in grafo.nodes]
cores = [comunidade_de[no] for no in grafo.nodes]
larguras = [0.5 + grafo[u][v]["peso"] * 0.6 for u, v in grafo.edges]
plt.figure(figsize=(10, 7))
nx.draw_networkx(
    grafo, posicao, node_size=tamanhos, node_color=cores, cmap="Set2",
    width=larguras, edge_color="#999999", font_size=8
)
plt.title("Grafo de coocorrência — dados inteiramente fictícios")
plt.axis("off"); plt.tight_layout(); plt.show()

## 3. Direção e atributos: correspondência não é coocorrência

Uma carta registra remetente e destinatário. Esse modelo requer um `DiGraph`:
grau de entrada e de saída respondem perguntas diferentes. Como a base contém
mais de um registro para alguns pares, primeiro agregamos as cartas por
remetente–destinatário e preservamos primeiro ano, último ano e número de registros.
Sem essa etapa, um `DiGraph` simples manteria apenas uma aresta por par e poderia
sobrescrever atributos anteriores.

In [ ]:
correspondencia = pd.read_csv("dados/correspondencia_direta.csv")
correspondencia_agregada = (
    correspondencia.groupby(["remetente", "destinatario"], as_index=False)
    .agg(
        cartas=("cartas", "sum"),
        primeiro_ano=("ano", "min"),
        ultimo_ano=("ano", "max"),
        registros=("ano", "size"),
    )
)
display(correspondencia_agregada)
grafo_dirigido = nx.from_pandas_edgelist(
    correspondencia_agregada,
    source="remetente", target="destinatario",
    edge_attr=["cartas", "primeiro_ano", "ultimo_ano", "registros"],
    create_using=nx.DiGraph,
)
graus_dirigidos = pd.DataFrame({
    "saída": dict(grafo_dirigido.out_degree()),
    "entrada": dict(grafo_dirigido.in_degree()),
    "cartas_enviadas": dict(grafo_dirigido.out_degree(weight="cartas")),
    "cartas_recebidas": dict(grafo_dirigido.in_degree(weight="cartas")),
}).fillna(0).astype(int)
display(graus_dirigidos.sort_values("cartas_recebidas", ascending=False))

O desenho de força é uma disposição, não um mapa geográfico nem uma medida. A
proximidade visual depende do algoritmo e da semente; leia métricas e documentos,
não apenas a aparência.

## U06-A05 — Atividade integrada — leitura estrutural e retorno aos casos

Escolha duas entidades com posições contrastantes. Compare grau, força e
intermediação; identifique suas arestas principais; recupere três documentos que
sustentam a interpretação; teste um limiar de peso e registre o que mudou.

**Resposta:** Escreva aqui.

## Passagem

As medidas ampliaram a descrição, mas não substituíram a crítica da relação. A
oficina exige escolher apenas o percurso pertinente ao projeto e incorporá-lo ao
mesmo relatório-notebook iniciado nas unidades anteriores.

**Leitura:** Freeman (1978/1979), Newman (2010), Wasserman e Faust (1994) e
Grandjean (2016), listados em `referencias.md`.